In [96]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("File loaded successfully:", filename)
print("Dataset shape:", df.shape)

display(df.head())

Saving Sales_Transactions_Dataset_Weekly.csv to Sales_Transactions_Dataset_Weekly (6).csv
File loaded successfully: Sales_Transactions_Dataset_Weekly (6).csv
Dataset shape: (811, 107)


,Product_Code,W0,W1,W2,W3,W4,W5,W6,W7,W8,...,Normalized 42,Normalized 43,Normalized 44,Normalized 45,Normalized 46,Normalized 47,Normalized 48,Normalized 49,Normalized 50,Normalized 51
0,P1,11,12,10,8,13,12,14,21,6,...,0.06,0.22,0.28,0.39,0.50,0.00,0.22,0.17,0.11,0.39
1,P2,7,6,3,2,7,1,6,3,3,...,0.20,0.40,0.50,0.10,0.10,0.40,0.50,0.10,0.60,0.00
2,P3,7,11,8,9,10,8,7,13,12,...,0.27,1.00,0.18,0.18,0.36,0.45,1.00,0.45,0.45,0.36
3,P4,12,8,13,5,9,6,9,13,13,...,0.41,0.47,0.06,0.12,0.24,0.35,0.71,0.35,0.29,0.35
4,P5,8,5,13,11,6,7,9,14,9,...,0.27,0.53,0.27,0.60,0.20,0.20,0.13,0.53,0.33,0.40


In [97]:
print(df.columns.tolist())

['Product_Code', 'W0', 'W1', 'W2', 'W3', 'W4', 'W5', 'W6', 'W7', 'W8', 'W9', 'W10', 'W11', 'W12', 'W13', 'W14', 'W15', 'W16', 'W17', 'W18', 'W19', 'W20', 'W21', 'W22', 'W23', 'W24', 'W25', 'W26', 'W27', 'W28', 'W29', 'W30', 'W31', 'W32', 'W33', 'W34', 'W35', 'W36', 'W37', 'W38', 'W39', 'W40', 'W41', 'W42', 'W43', 'W44', 'W45', 'W46', 'W47', 'W48', 'W49', 'W50', 'W51', 'MIN', 'MAX', 'Normalized 0', 'Normalized 1', 'Normalized 2', 'Normalized 3', 'Normalized 4', 'Normalized 5', 'Normalized 6', 'Normalized 7', 'Normalized 8', 'Normalized 9', 'Normalized 10', 'Normalized 11', 'Normalized 12', 'Normalized 13', 'Normalized 14', 'Normalized 15', 'Normalized 16', 'Normalized 17', 'Normalized 18', 'Normalized 19', 'Normalized 20', 'Normalized 21', 'Normalized 22', 'Normalized 23', 'Normalized 24', 'Normalized 25', 'Normalized 26', 'Normalized 27', 'Normalized 28', 'Normalized 29', 'Normalized 30', 'Normalized 31', 'Normalized 32', 'Normalized 33', 'Normalized 34', 'Normalized 35', 'Normalized 3

In [98]:
# Select weekly sales columns
weekly_columns = [f'W{i}' for i in range(52)]

sales_data = df[weekly_columns].copy()

print("Weekly sales data shape:", sales_data.shape)
display(sales_data.head())

Weekly sales data shape: (811, 52)


,W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,...,W42,W43,W44,W45,W46,W47,W48,W49,W50,W51
0,11,12,10,8,13,12,14,21,6,14,...,4,7,8,10,12,3,7,6,5,10
1,7,6,3,2,7,1,6,3,3,3,...,2,4,5,1,1,4,5,1,6,0
2,7,11,8,9,10,8,7,13,12,6,...,6,14,5,5,7,8,14,8,8,7
3,12,8,13,5,9,6,9,13,13,11,...,9,10,3,4,6,8,14,8,7,8
4,8,5,13,11,6,7,9,14,9,9,...,7,11,7,12,6,6,5,11,8,9


In [99]:
# Create and fit the K-Means model
kmeans = KMeans(n_clusters=8, random_state=8, n_init=10)

kmeans.fit(sales_data)

# Get cluster labels
labels = kmeans.labels_

print("Number of clusters:", len(set(labels)))
print("Number of labels:", len(labels))

Number of clusters: 8
Number of labels: 811


In [100]:
# Create a DataFrame with weekly sales and cluster labels
clustered_data = sales_data.copy()
clustered_data["label"] = labels

print("Clustered data shape:", clustered_data.shape)
display(clustered_data.head())

Clustered data shape: (811, 53)


,W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,...,W43,W44,W45,W46,W47,W48,W49,W50,W51,label
0,11,12,10,8,13,12,14,21,6,14,...,7,8,10,12,3,7,6,5,10,0
1,7,6,3,2,7,1,6,3,3,3,...,4,5,1,1,4,5,1,6,0,5
2,7,11,8,9,10,8,7,13,12,6,...,14,5,5,7,8,14,8,8,7,0
3,12,8,13,5,9,6,9,13,13,11,...,10,3,4,6,8,14,8,7,8,0
4,8,5,13,11,6,7,9,14,9,9,...,11,7,12,6,6,5,11,8,9,0


In [101]:
# Count products in each cluster
cluster_counts = clustered_data.groupby("label").size()

# Calculate average weekly sales for each product
clustered_data["yearly_average_sales"] = (
    clustered_data[weekly_columns].sum(axis=1)
)

# Summarize each cluster
cluster_summary = clustered_data.groupby("label").agg(
    count=("yearly_average_sales", "size"),
    total_sales=("yearly_average_sales", "sum"),
    yearly_average_sales=("yearly_average_sales", "mean")
)

print("Cluster summary:")
display(cluster_summary.round(2))

Cluster summary:


,count,total_sales,yearly_average_sales
label,,,
0,114,54520,478.25
1,70,117171,1673.87
2,302,6364,21.07
3,44,41086,933.77
4,42,26065,620.60
5,188,35971,191.34
6,50,91890,1837.80
7,1,2220,2220.00


# Activity 5.01: Clustering Sales Data Using K-Means

## Objective

The objective of this activity is to group products according to their weekly sales patterns using the K-Means clustering algorithm.

## Dataset Overview

The dataset contains weekly sales information for 811 products over 52 weeks. The weekly sales columns, from W0 to W51, are selected as the input features for clustering.

## K-Means Clustering

The K-Means algorithm is configured to create eight clusters using `random_state=8` to make the results reproducible. Each product is assigned a cluster label.

## Cluster Summary

After clustering, the products are grouped by their cluster labels. The summary reports the number of products, total annual sales, and average annual sales per product in each cluster.